In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

/opt/conda/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")

In [5]:
bart_ridership['Day Type'] = bart_ridership.apply(
    lambda row: 'Weekday' if row['Day Type'] == 'Holiday' and row['Day of Week'] in ['Monday','Tuesday','Wednesday','Thursday','Friday']
    else ('Saturday' if row['Day Type'] == 'Holiday' and row['Day of Week'] == 'Saturday'
          else ('Sunday' if row['Day Type'] == 'Holiday' and row['Day of Week'] == 'Sunday'
                else row['Day Type'])),
    axis=1
)

In [6]:
# Add exposure column
def calculate_exposure(row):
    start = row['start_date']
    end = row['end_date']
    day_type = row['Day Type'].upper()  # 'WEEKDAY', 'SATURDAY', 'SUNDAY'
    
    # Create date range
    date_range = pd.date_range(start, end)
    
    if day_type == 'WEEKDAY':
        return sum(date.weekday() < 5 for date in date_range)  # Mon-Fri
    elif day_type == 'SATURDAY':
        return sum(date.weekday() == 5 for date in date_range)
    elif day_type == 'SUNDAY':
        return sum(date.weekday() == 6 for date in date_range)
    else:
        return np.nan

bart_ridership['exposure'] = bart_ridership.apply(calculate_exposure, axis=1)

In [10]:
bart_ridership.rename(columns={
    'Station': 'Stop',
    'Station Name': 'Stop Name',
    'Entries': 'Boardings'
}, inplace=True)

In [11]:
bart_ridership['Day Type'] = bart_ridership['Day Type'].astype('category')

In [19]:
big_blue_bus_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11163 entries, 0 to 11162
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   SERVICE_PERIOD            11163 non-null  datetime64[ns]
 1   SERVICE_DAY               11163 non-null  object        
 2   ROUTE_NUMBER              11163 non-null  int64         
 3   ROUTE_NAME                11163 non-null  object        
 4   DIRECTION_NAME            11163 non-null  object        
 5   STOP_ID                   11163 non-null  int64         
 6   STOP_NAME                 11163 non-null  object        
 7   STOP_LAT                  11163 non-null  float64       
 8   STOP_LON                  11163 non-null  float64       
 9   AVERAGE_DAILY_BOARDINGS   11163 non-null  float64       
 10  AVERAGE_DAILY_ALIGHTINGS  11163 non-null  float64       
 11  start_date                11163 non-null  datetime64[ns]
 12  end_date          

In [21]:
# Function to count number of days of a Day_Type in a service period
def count_day_type_days(start, end, day_type):
    dates = pd.date_range(start, end)
    day_type_upper = day_type.upper()
    if day_type_upper == 'WEEKDAY':
        return sum(d.weekday() < 5 for d in dates)  # Mon-Fri
    elif day_type_upper == 'SATURDAY':
        return sum(d.weekday() == 5 for d in dates)
    elif day_type_upper == 'SUNDAY':
        return sum(d.weekday() == 6 for d in dates)
    else:
        return 0

# Compute exposure: number of days of that Day_Type in the period
big_blue_bus_ridership['exposure'] = big_blue_bus_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['SERVICE_DAY']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
big_blue_bus_ridership['Boardings'] = big_blue_bus_ridership['AVERAGE_DAILY_BOARDINGS'] * big_blue_bus_ridership['exposure']

# Rename columns to match BART dataset
big_blue_bus_ridership.rename(columns={
    'SERVICE_DAY': 'Day_Type',
    'STOP_ID': 'Stop',
    'STOP_NAME': 'Stop Name'
}, inplace=True)


In [24]:
big_blue_bus_ridership.sample(5)

,SERVICE_PERIOD,Day_Type,ROUTE_NUMBER,ROUTE_NAME,DIRECTION_NAME,Stop,Stop Name,STOP_LAT,STOP_LON,AVERAGE_DAILY_BOARDINGS,AVERAGE_DAILY_ALIGHTINGS,start_date,end_date,exposure,Boardings
5903,2025-04-01,SATURDAY,7,Pico Blvd,EASTBOUND,2788,CRENSHAW NB/PICO FS,34.048124,-118.326428,23.916811,171.913454,2025-04-01,2025-07-31,17,406.585781
8484,2025-08-01,SATURDAY,3,Lincoln Blvd/LAX,SOUTHBOUND,2776,LINCOLN SB/JEFFERSON FS,33.971527,-118.430187,24.933332,38.699998,2025-08-01,2025-11-30,18,448.799980
1743,2024-08-01,WEEKDAY,3,Lincoln Blvd/LAX,SOUTHBOUND,2225,LINCOLN SB/MARINA POINTE FS,33.984964,-118.443180,69.418739,86.013704,2024-08-01,2024-11-30,87,6039.430329
7915,2025-04-01,WEEKDAY,17,UCLA-VA Medical Center-Palms,NORTHBOUND,2249,SAWTELLE NB/OHIO FS,34.048674,-118.450235,33.070315,15.866342,2025-04-01,2025-07-31,88,2910.187756
7126,2025-04-01,WEEKDAY,1,Main St & Santa Monica Blvd/UCLA,WESTBOUND,2132,SANTA MONICA WB/BARRINGTON FS,34.043685,-118.456528,65.325428,57.581025,2025-04-01,2025-07-31,88,5748.637705
